In [1]:
from netgen.occ import *
import ngsolve
from ngsolve import *
from ngsolve.webgui import Draw
import numpy as np
from tqdm import tqdm

In [2]:
pfeiler=Box((-0.5,-1.5,-15), (1.5,2.5,60))
strebe=Box((1,0.2,33),(11,0.8,38))

road=Box((0.5,-5,10),(10.5,105,10.5))

strebe.faces.name="fix"

pfeiler.faces.name="fix"

road.faces.Min(Y).name = "fix"
road.faces.Max(Y).name = "fix"

road.faces.Max(Z).name = "force"

geo=pfeiler+pfeiler.Move((10,0,0))+pfeiler.Move((10,100,0))+pfeiler.Move((0,100,0))
geo+=strebe+strebe.Move((0,0,20))+strebe.Move((0,100,20))+strebe.Move((0,100,0))
geo+=road


seil1=Box((0.5,10-0.25,10), (0.75,10.25,45.2))
seil2=Box((0.5,10-0.25+10,10), (0.75,10.25+10,37.2))
seil3=Box((0.5,10-0.25+20,10), (0.75,10.25+20,32.3))
seil4=Box((0.5,10-0.25+30,10), (0.75,10.25+30,29.5))
seil5=Box((0.5,10-0.25+40,10), (0.75,10.25+40,28.6))

seile = WorkPlane().Line(0.8).Rotate(90).Line(0.8).Rotate(90).Line(0.8).Rotate(90).Line(0.8).Face()
bent_cylinder = seile.Revolve(Axis((10,56,0),X),126)

bent_cylinder1=bent_cylinder.Rotate(Axis((0,0,0),X),26.5).Move((0.1,0,59))

geo += bent_cylinder1+bent_cylinder1.Move((10,0,0))
#geo+=seil1+seil1.Move((9.75,0,0))+seil1.Move((9.75,80,0))+seil1.Move((0,80,0))
geo+=seil2+seil2.Move((9.75,0,0))+seil2.Move((9.75,60,0))+seil2.Move((0,60,0))
#geo+=seil3+seil3.Move((9.75,0,0))+seil3.Move((9.75,40,0))+seil3.Move((0,40,0))
geo+=seil4+seil4.Move((9.75,0,0))+seil4.Move((9.75,20,0))+seil4.Move((0,20,0))
geo+=seil5+seil5.Move((9.75,0,0))

Draw(geo, height="3vh")

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'ngsolve_version': 'Netgen x.x', 'mesh_dim': 3,…

BaseWebGuiScene

In [3]:
from ngsolve import *
from ngsolve.webgui import Draw
mesh = Mesh(OCCGeometry(geo).GenerateMesh(maxh=3)).Curve(3)
Draw(mesh, height="3vh");

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.240…

In [4]:
dt = 0.1
beta = .25 # 1/4 limit for unconditional stability, set higher for more damping
gamma = .5 # trapezoidal rule

E, nu = 400, 0.2
mu  = E / 2 / (1+nu)
lam = E * nu / ((1+nu)*(1-2*nu))

def Stress(strain):
    return 2*mu*strain + lam*Trace(strain)*Id(3)

In [5]:
t = Parameter(0.0)
fes = VectorH1(mesh, order=3, dirichlet="fix")
u,v = fes.TnT()
gfu = GridFunction(fes)
gfv = GridFunction(fes)
gfa = GridFunction(fes)

with TaskManager():
    m_int = u*v*dx
    m = BilinearForm(m_int)
    m.Assemble()
    a_int = InnerProduct(Stress(Sym(Grad(u))), Sym(Grad(v))).Compile()*dx
    a = BilinearForm(a_int)
    a.Assemble()
    # amp1 = 1
    # amp2 = 1
    # c_int = amp1*u*v*dx + amp2*grad(u)*v*dx
    # c = BilinearForm(amp1* + amp2*)
    # c.Assemble()
    mstar = BilinearForm(fes)
    mstar += m_int
    #mstar += gamma * dt * c_int
    mstar += beta * dt**2 * a_int
    pre = MultiGridPreconditioner(mstar, "bddc") # Adds factor 8 Speedup for some reason ...
    mstar.Assemble()
    

In [6]:
omega = (0.7/2) # resoncance with  is around 1.5
force = CF((1,0,-0.3))*sin(2*np.pi*omega*t)
f = LinearForm(force*v*ds("force"))

In [7]:
from ngsolve.krylovspace import CGSolver

acc_inv = CGSolver(mstar.mat, pre, printrates=False, tol=1e-8)

$$ a^{n} = \left(\frac{\partial^2 u}{\partial t^2}\right)^{n} = \mathrm{gfa^{n}} $$
$$ v^{n} = \left(\frac{\partial u}{\partial t}\right)^{n} = \mathrm{gfv^{n}} $$

So what we do here ([Wikipedia](https://en.wikipedia.org/wiki/Newmark-beta_method)):



In [8]:
gfa.Set(CF((0,0,0)))
gfv.Set(CF((0,0,0)))
gfu.Set(CF((0,0,0)))

In [9]:
gfut = GridFunction(gfu.space,multidim=0)

fesstress = MatrixValued(H1(mesh,order=3), symmetric=True)
gfstress = GridFunction(fesstress)
gfstresst = GridFunction(fesstress, multidim=0)

tend = t.Get()+7
steps = int((tend - t.Get()) / dt)

with TaskManager():
    for _ in tqdm(range(steps), desc="Time-stepping", ncols=100, unit="step"):
        f.Assemble()

        rhs_acceleration = f.vec - a.mat*(gfu.vec + dt*gfv.vec + .5*dt**2*(1-2*beta)*gfa.vec) # - c.mat*(gfv.vec + (1-gamma)*gfa.vec) 
        
        new_acceleration = acc_inv * rhs_acceleration
        
        gfu.vec.data = gfu.vec + dt*gfv.vec + 0.5*dt**2*((1-2*beta)*gfa.vec + 2*beta*new_acceleration)
        
        gfv.vec.data = gfv.vec + dt*((1-gamma)*gfa.vec + gamma*new_acceleration)
        
        gfa.vec.data = new_acceleration
        
        gfut.AddMultiDimComponent(gfu.vec)
        
        gfstress.Interpolate(Stress(Sym(Grad(gfu))))
        
        gfstresst.AddMultiDimComponent(gfstress.vec)
        
        t.Set(t.Get()+dt)
    

Time-stepping: 100%|██████████████████████████████████████████████| 70/70 [02:22<00:00,  2.03s/step]


In [10]:
with TaskManager():
    fesstress = MatrixValued(H1(mesh,order=3), symmetric=True)
    gfstress = GridFunction(fesstress)
    gfstress.Interpolate (Stress(Sym(Grad(gfu))))

In [11]:
Draw (gfut, mesh, height="3vh", deformation=gfut, interpolate_multidim=True, animate=True)

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.240…

BaseWebGuiScene

In [12]:
Draw(gfstresst, mesh, height="3vh", deformation=gfut, interpolate_multidim=True, animate=True);

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.240…